In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
from xgboost import XGBRegressor
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# TensorFlow تنظیمات
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

# ============================================================================
# بخش 1: تعریف توابع تحلیل (5 تابع - سری اول: پایش انحراف بیرینگ)
# ============================================================================

def analysis_1_xgboost_wide(file_path, output_filename):
    """
    تحلیل پایش انحراف با XGBoost - فرمت Wide - کد شماره 1
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 1 (XGBoost - Wide Format)")
    print(f"{'='*60}")
    
    all_features = [
        'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
        'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_9357',
        'AssetID_9343', 'AssetID_9344', 'AssetID_9408'
    ]
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    try:
        df_raw = pd.read_excel(file_path)
        print(f"✅ فایل اصلی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {}
    for cluster_id in set(labels):
        if cluster_id != -1:
            cluster_centers[cluster_id] = scaled_data[labels == cluster_id].mean(axis=0)
    
    def calculate_distance(row_index):
        label = labels[row_index]
        point = scaled_data[row_index].reshape(1, -1)
        if label != -1:
            center = cluster_centers[label].reshape(1, -1)
            return cdist(point, center, metric='euclidean')[0][0]
        else:
            if not cluster_centers: return 0.0
            all_centers = np.array(list(cluster_centers.values()))
            return np.min(cdist(point, all_centers, metric='euclidean'))
    
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    
    before_count = len(df_raw)
    df_sorted = df_raw.sort_values(by='distance', ascending=False)
    num_to_remove = int(len(df_sorted)*0.10)
    df_cleaned = df_sorted.iloc[num_to_remove:].copy()
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
    df_cleaned = df_cleaned.sort_values(by='date')
    
    split_idx = int(len(df_cleaned) * 0.80)
    train_df = df_cleaned.iloc[:split_idx].copy()
    test_df = df_cleaned.iloc[split_idx:].copy()
    
    print(f"   داده‌های آموزش: {len(train_df):,} رکورد")
    print(f"   داده‌های تست: {len(test_df):,} رکورد")
    
    print("🔄 مرحله 2: یادگیری و پیش‌بینی برای تمام سنسورها...")
    
    for target in target_sensors:
        current_features = [f for f in all_features if f != target]
        X_train, y_train = train_df[current_features], train_df[target]
        X_test = test_df[current_features]
        
        model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbosity=0)
        model.fit(X_train, y_train)
        
        column_name = f'predicted_{target.split("_")[1]}'
        test_df[column_name] = model.predict(X_test)
        
        mse = np.mean((test_df[target] - test_df[column_name]) ** 2)
        rmse = np.sqrt(mse)
        print(f"   ✅ {target}: RMSE = {rmse:.6f}")
    
    predicted_cols = [f'predicted_{t.split("_")[1]}' for t in target_sensors]
    final_columns_order = ['date'] + all_features + predicted_cols
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        test_df[final_columns_order].to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        print(f"📊 تعداد رکوردها: {len(test_df):,}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_2_xgboost_long(file_path, output_filename):
    """
    تحلیل پایش انحراف با XGBoost - فرمت Long - کد شماره 2
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 2 (XGBoost - Long Format)")
    print(f"{'='*60}")
    
    all_features = [
        'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
        'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_9357',
        'AssetID_9343', 'AssetID_9344', 'AssetID_9408'
    ]
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    try:
        df_raw = pd.read_excel(file_path)
        print(f"✅ فایل اصلی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {i: scaled_data[labels == i].mean(axis=0) for i in set(labels) if i != -1}
    
    def calc_dist(idx):
        label = labels[idx]
        point = scaled_data[idx].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calc_dist(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
    df_cleaned = df_cleaned.sort_values(by='date')
    
    split_idx = int(len(df_cleaned) * 0.80)
    train_df = df_cleaned.iloc[:split_idx].copy()
    test_df = df_cleaned.iloc[split_idx:].copy()
    
    print(f"   داده‌های آموزش: {len(train_df):,} رکورد")
    print(f"   داده‌های تست: {len(test_df):,} رکورد")
    
    print("🔄 مرحله 2: یادگیری و پیش‌بینی برای تمام سنسورها...")
    
    for target in target_sensors:
        features = [f for f in all_features if f != target]
        X_train, y_train = train_df[features], train_df[target]
        
        model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbosity=0)
        model.fit(X_train, y_train)
        
        test_df[f'PRED_{target}'] = model.predict(test_df[features])
        
        mse = np.mean((test_df[target] - test_df[f'PRED_{target}']) ** 2)
        rmse = np.sqrt(mse)
        print(f"   ✅ {target}: RMSE = {rmse:.6f}")
    
    print("🔄 مرحله 3: تغییر ساختار داده به فرمت Long...")
    
    melted_actual = test_df.melt(id_vars=['date'], value_vars=target_sensors, 
                                 var_name='AssetID', value_name='actual')
    
    pred_cols = [f'PRED_{t}' for t in target_sensors]
    melted_pred = test_df.melt(id_vars=['date'], value_vars=pred_cols, 
                               var_name='temp_target', value_name='predicted')
    melted_pred['AssetID'] = melted_pred['temp_target'].str.replace('PRED_', '')
    
    final_long_df = pd.merge(melted_actual, melted_pred[['date', 'AssetID', 'predicted']], 
                             on=['date', 'AssetID'])
    final_long_df['error'] = final_long_df['actual'] - final_long_df['predicted']
    final_long_df['abs_error'] = np.abs(final_long_df['error'])
    
    print(f"   ✅ تعداد رکوردهای نهایی: {len(final_long_df):,}")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_long_df.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        print(f"📊 تعداد رکوردها: {len(final_long_df):,}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_3_xgboost_refined(file_path, output_filename):
    """
    تحلیل پایش انحراف با XGBoost + DBSCAN پالایش - کد شماره 3
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 3 (XGBoost + DBSCAN Refined)")
    print(f"{'='*60}")
    
    all_sensors = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 
                   'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل اصلی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    print("🔄 مرحله 1: جداسازی داده‌های آموزشی و تست...")
    
    last_date = df_raw['date'].max()
    split_date = last_date - pd.Timedelta(days=20)
    raw_train_set = df_raw[df_raw['date'] <= split_date].copy()
    test_set = df_raw[df_raw['date'] > split_date].copy()
    
    print(f"   داده‌های آموزشی: {len(raw_train_set):,} رکورد")
    print(f"   داده‌های تست: {len(test_set):,} رکورد")
    
    print("🔄 مرحله 2: پالایش داده‌های آموزشی با DBSCAN...")
    
    scaler = StandardScaler()
    scaled_train = scaler.fit_transform(raw_train_set[all_sensors])
    
    dbscan = DBSCAN(eps=0.8, min_samples=10)
    labels = dbscan.fit_predict(scaled_train)
    
    normal_train_set = raw_train_set[labels != -1].copy()
    print(f"   داده‌های سالم تایید شده: {len(normal_train_set):,} رکورد")
    print(f"   داده‌های پرت حذف شده: {len(raw_train_set) - len(normal_train_set):,} ردیف")
    
    print("🔄 مرحله 3: آموزش مدل‌های XGBoost...")
    
    models = {}
    for target in all_sensors:
        features = [s for s in all_sensors if s != target]
        model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbosity=0)
        model.fit(normal_train_set[features], normal_train_set[target])
        models[target] = model
        print(f"   ✅ مدل برای {target} آموزش داده شد")
    
    print("🔄 مرحله 4: پایش انحراف در داده‌های تست...")
    
    all_errors = pd.DataFrame(index=test_set.index)
    for target in all_sensors:
        features = [s for s in all_sensors if s != target]
        preds = models[target].predict(test_set[features])
        all_errors[f'err_{target}'] = np.abs(test_set[target] - preds)
    
    test_set['System_Deviation_Index'] = all_errors.mean(axis=1)
    
    train_preds_all = []
    for target in all_sensors:
        features = [s for s in all_sensors if s != target]
        train_preds = models[target].predict(normal_train_set[features])
        train_preds_all.append(np.abs(normal_train_set[target] - train_preds))
    
    mean_train_error = np.mean(train_preds_all)
    std_train_error = np.std(train_preds_all)
    threshold = mean_train_error + (3*std_train_error)
    
    test_set['Is_Anomaly'] = test_set['System_Deviation_Index'] > threshold
    test_set['Anomaly_Label'] = test_set['Is_Anomaly'].map({True: '⚠️ Anomaly', False: '✅ Normal'})
    
    anomaly_count = test_set['Is_Anomaly'].sum()
    print(f"\n📊 تعداد ناهنجاری‌ها: {anomaly_count:,} ({anomaly_count/len(test_set)*100:.2f}%)")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        test_set.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_4_autoencoder_basic(file_path, output_filename):
    """
    تحلیل ناهنجاری با Autoencoder - کد شماره 4
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 4 (Autoencoder - Basic)")
    print(f"{'='*60}")
    
    all_sensors = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 
                   'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']
    
    try:
        df = pd.read_excel(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(by='date')
        print(f"✅ فایل اصلی خوانده شد. تعداد رکوردها: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    df_clean = df.dropna(subset=all_sensors).copy()
    raw_data = df_clean[all_sensors].values
    print(f"📊 تعداد رکوردهای بدون داده خالی: {len(df_clean):,}")
    
    print("🔄 مرحله 1: نرمال‌سازی داده‌ها...")
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(raw_data)
    print(f"   ✅ نرمال‌سازی با {len(all_sensors)} سنسور انجام شد")
    
    print("🔄 مرحله 2: ساخت و آموزش مدل Autoencoder...")
    
    input_dim = len(all_sensors)
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(16, activation='relu')(input_layer)
    latent_space = Dense(8, activation='relu')(encoded)
    decoded = Dense(16, activation='relu')(latent_space)
    output_layer = Dense(input_dim, activation='sigmoid')(decoded)
    
    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
    
    print("   🚀 در حال یادگیری رفتار سیستم...")
    autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)
    print("   ✅ آموزش مدل با موفقیت انجام شد")
    
    print("🔄 مرحله 3: محاسبه خطای بازسازی...")
    reconstructed_data = autoencoder.predict(scaled_data, verbose=0)
    mse_errors = np.mean(np.power(scaled_data - reconstructed_data, 2), axis=1)
    
    threshold = np.mean(mse_errors) + (3*np.std(mse_errors))
    print(f"   ✅ آستانه ناهنجاری: {threshold:.6f}")
    
    df_clean['Systematic_Deviation_Index'] = mse_errors
    df_clean['Anomaly_Threshold'] = threshold
    df_clean['Is_Anomaly'] = df_clean['Systematic_Deviation_Index'] > threshold
    df_clean['Anomaly_Label'] = df_clean['Is_Anomaly'].map({True: '⚠️ Anomaly', False: '✅ Normal'})
    
    print("🔄 مرحله 4: فیلتر کردن داده‌های یک ماه اخیر...")
    
    last_date = df_clean['date'].max()
    start_of_last_month = last_date - pd.Timedelta(days=30)
    df_last_month = df_clean[df_clean['date'] >= start_of_last_month].copy()
    
    print(f"   تعداد رکوردهای ماه اخیر: {len(df_last_month):,}")
    print(f"   تعداد ناهنجاری: {df_last_month['Is_Anomaly'].sum():,}")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        df_last_month.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_5_autoencoder_advanced(file_path, output_filename):
    """
    تحلیل ناهنجاری با Autoencoder پیشرفته + EWMA + آستانه‌های پویا - کد شماره 5
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 5 (Autoencoder + EWMA + Dynamic Thresholds)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 
                    'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']
    
    try:
        df = pd.read_excel(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(by='date')
        print(f"✅ فایل اصلی خوانده شد. تعداد رکوردها: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False
    
    df_clean = df.dropna(subset=all_features).copy()
    raw_data = df_clean[all_features].values
    print(f"📊 تعداد رکوردهای بدون داده خالی: {len(df_clean):,}")
    
    print("🔄 مرحله 1: نرمال‌سازی داده‌ها...")
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(raw_data)
    print(f"   ✅ نرمال‌سازی با {len(all_features)} سنسور انجام شد")
    
    print("🔄 مرحله 2: ساخت و آموزش مدل Autoencoder...")
    
    input_dim = len(all_features)
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(16, activation='relu')(input_layer)
    latent = Dense(8, activation='relu')(encoded)
    decoded = Dense(16, activation='relu')(latent)
    output_layer = Dense(input_dim, activation='sigmoid')(decoded)
    
    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
    
    print("   🚀 یادگیری رفتار سیستماتیک سنسورها...")
    autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)
    print("   ✅ آموزش مدل با موفقیت انجام شد")
    
    print("🔄 مرحله 3: محاسبه شاخص‌های انحراف...")
    
    reconstructed = autoencoder.predict(scaled_data, verbose=0)
    mse_errors = np.mean(np.power(scaled_data - reconstructed, 2), axis=1)
    df_clean['Raw_Anomaly_Index'] = mse_errors
    
    print("🔄 مرحله 4: ارزیابی پیشرفته باقیمانده‌ها...")
    
    df_clean['Smooth_Deviation_Index'] = df_clean['Raw_Anomaly_Index'].ewm(alpha=0.1).mean()
    
    p95_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.95)
    p99_threshold = df_clean['Smooth_Deviation_Index'].quantile(0.99)
    
    print(f"   📈 آستانه هشدار (زرد): {p95_threshold:.6f}")
    print(f"   🚫 آستانه بحرانی (قرمز): {p99_threshold:.6f}")
    
    def final_judgment(val):
        if val > p99_threshold:
            return "Critical (Red) - Systematic Failure"
        elif val > p95_threshold:
            return "Warning (Yellow) - Operational Drift"
        else:
            return "Normal (Green)"
    
    df_clean['Final_Status'] = df_clean['Smooth_Deviation_Index'].apply(final_judgment)
    
    print("🔄 مرحله 5: فیلتر کردن داده‌های یک ماه اخیر...")
    
    last_date = df_clean['date'].max()
    start_of_last_month = last_date - pd.Timedelta(days=30)
    df_output = df_clean[df_clean['date'] >= start_of_last_month].copy()
    
    print(f"   تعداد رکوردهای ماه اخیر: {len(df_output):,}")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        df_output.to_excel(output_filename, index=False)
        print(f"✅ فایل ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# بخش 2: تعریف وظایف (Jobs) - 5 وظیفه سری اول
# ============================================================================

def get_analysis_jobs():
    """
    تعریف ۵ وظیفه تحلیل - سری اول: پایش انحراف بیرینگ
    """
    base_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
    out_base = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output'
    
    jobs = [
        {
            'name': 'Analysis 1 - XGBoost Wide',
            'function': analysis_1_xgboost_wide,
            'file_path': base_path,
            'output_filename': f'{out_base}1.xlsx'
        },
        {
            'name': 'Analysis 2 - XGBoost Long',
            'function': analysis_2_xgboost_long,
            'file_path': base_path,
            'output_filename': f'{out_base}2.xlsx'
        },
        {
            'name': 'Analysis 3 - XGBoost Refined',
            'function': analysis_3_xgboost_refined,
            'file_path': base_path,
            'output_filename': f'{out_base}3.xlsx'
        },
        {
            'name': 'Analysis 4 - Autoencoder Basic',
            'function': analysis_4_autoencoder_basic,
            'file_path': base_path,
            'output_filename': f'{out_base}4.xlsx'
        },
        {
            'name': 'Analysis 5 - Autoencoder Advanced',
            'function': analysis_5_autoencoder_advanced,
            'file_path': base_path,
            'output_filename': f'{out_base}5.xlsx'
        }
    ]
    return jobs


def run_all_analyses():
    """
    اجرای تمام ۵ تحلیل به ترتیب
    """
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌های سری اول (۵ وظیفه)")
    print(f"📅 زمان: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    print("📋 لیست تحلیلها:")
    print("   1. XGBoost - Wide Format")
    print("   2. XGBoost - Long Format")
    print("   3. XGBoost + DBSCAN Refined")
    print("   4. Autoencoder - Basic")
    print("   5. Autoencoder + EWMA + Dynamic Thresholds")
    print("="*80)
    
    jobs = get_analysis_jobs()
    results = []
    
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# اجرای وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"{'#'*80}")
        
        try:
            success = job['function'](job['file_path'], job['output_filename'])
            results.append({
                'job_name': job['name'],
                'success': success,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            if success:
                print(f"✅ وظیفه {i} با موفقیت کامل شد")
            else:
                print(f"❌ وظیفه {i} با شکست مواجه شد")
                
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({
                'job_name': job['name'],
                'success': False,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'error': str(e)
            })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    for r in results:
        status = "✅" if r['success'] else "❌"
        print(f"   {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 3: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)
    """
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - سری اول: پایش انحراف بیرینگ")
    print("="*80)
    print("📋 شامل ۵ تحلیل:")
    print("   1. XGBoost - Wide Format")
    print("   2. XGBoost - Long Format")
    print("   3. XGBoost + DBSCAN Refined")
    print("   4. Autoencoder - Basic")
    print("   5. Autoencoder + EWMA + Dynamic Thresholds")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            if current_time in ["15:12", "21:00"]:
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    results = run_all_analyses()
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    time.sleep(60)
            
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            time.sleep(60)


# ============================================================================
# بخش 4: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه سری اول - پایش انحراف بیرینگ (۵ تحلیل یکپارچه)")
        print("="*80)
        print("📋 لیست تحلیلها:")
        print("   1. XGBoost - Wide Format (۱ خروجی)")
        print("   2. XGBoost - Long Format (۱ خروجی)")
        print("   3. XGBoost + DBSCAN Refined (۱ خروجی)")
        print("   4. Autoencoder - Basic (۱ خروجی)")
        print("   5. Autoencoder + EWMA + Dynamic Thresholds (۱ خروجی)")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")


🚀 شروع برنامه سری اول - پایش انحراف بیرینگ (۵ تحلیل یکپارچه)
📋 لیست تحلیلها:
   1. XGBoost - Wide Format (۱ خروجی)
   2. XGBoost - Long Format (۱ خروجی)
   3. XGBoost + DBSCAN Refined (۱ خروجی)
   4. Autoencoder - Basic (۱ خروجی)
   5. Autoencoder + EWMA + Dynamic Thresholds (۱ خروجی)
⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - سری اول: پایش انحراف بیرینگ
📋 شامل ۵ تحلیل:
   1. XGBoost - Wide Format
   2. XGBoost - Long Format
   3. XGBoost + DBSCAN Refined
   4. Autoencoder - Basic
   5. Autoencoder + EWMA + Dynamic Thresholds
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-12 15:12:58

🚀 شروع اجرای همه تحلیل‌های سری اول (۵ وظیفه)
📅 زمان: 2026-07-12 15:12:58
📋 لیست تحلیلها:
   1. XGBoost - Wide Format
   2. XGBoost - Long Format
   3. XGBoost + DBSCAN Refined
   4. Autoencoder - Basic
   5. Autoencoder + EWMA + Dynamic Thresholds

########################################